In [20]:
import cv2
import numpy as np
import os

input_folder  = "SHARP_MC"
output_folder = "SEGMENTATION_RESULTS"
os.makedirs(output_folder, exist_ok=True)

for file in sorted(os.listdir(input_folder)):
    if not file.lower().endswith((".jpg", ".jpeg", ".png")):
        continue

    image = cv2.imread(os.path.join(input_folder, file))
    if image is None:
        continue

    name = os.path.splitext(file)[0]
    h, w = image.shape[:2]
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    # STEP 1: Blur
    blur = cv2.GaussianBlur(gray, (7, 7), 0)

    # STEP 2: Canny edges
    edges = cv2.Canny(blur, 60, 140)

    # STEP 3: Tight ROI — only the pothole line band (28%–44% from top)
    mask = np.zeros_like(edges)
    mask[int(h * 0.28):int(h * 0.44), :] = 255
    edges = cv2.bitwise_and(edges, mask)

    # STEP 4: Morphology — connect broken edges
    kernel = np.ones((3, 3), np.uint8)
    morph  = cv2.morphologyEx(edges, cv2.MORPH_CLOSE, kernel, iterations=2)

    # STEP 5: Contours
    contours, _ = cv2.findContours(morph, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    detected = image.copy()
    count = 0

    for cnt in contours:
        area = cv2.contourArea(cnt)
        x, y, bw, bh = cv2.boundingRect(cnt)
        aspect = bw / float(bh + 1)
        if area > 20 and aspect < 4:
            cv2.rectangle(detected, (x, y), (x+bw, y+bh), (0, 0, 255), 2)
            count += 1

    # Save 3 separate outputs
    cv2.imwrite(os.path.join(output_folder, f"{name}_edges.jpg"),     edges)
    cv2.imwrite(os.path.join(output_folder, f"{name}_segmented.jpg"), morph)
    cv2.imwrite(os.path.join(output_folder, f"{name}_detected.jpg"),  detected)

    print(f"{file} → {count} potholes detected")

print("✅ Done!")

frame_0.png → 5 potholes detected
frame_1.png → 6 potholes detected
frame_10.png → 2 potholes detected
frame_11.png → 1 potholes detected
frame_12.png → 0 potholes detected
frame_13.png → 0 potholes detected
frame_14.png → 1 potholes detected
frame_15.png → 2 potholes detected
frame_16.png → 1 potholes detected
frame_17.png → 8 potholes detected
frame_18.png → 6 potholes detected
frame_19.png → 5 potholes detected
frame_2.png → 4 potholes detected
frame_20.png → 1 potholes detected
frame_21.png → 0 potholes detected
frame_22.png → 0 potholes detected
frame_23.png → 0 potholes detected
frame_24.png → 1 potholes detected
frame_25.png → 1 potholes detected
frame_26.png → 2 potholes detected
frame_27.png → 4 potholes detected
frame_28.png → 2 potholes detected
frame_29.png → 3 potholes detected
frame_3.png → 5 potholes detected
frame_30.png → 3 potholes detected
frame_31.png → 4 potholes detected
frame_32.png → 5 potholes detected
frame_33.png → 2 potholes detected
frame_34.png → 2 pothole